# dbt Models Notebook
## Purpose: Run dbt to build analytics marts from Silver tables

**Note: Uses Spark tables as dbt sources (DBFS disabled in Free Edition)**

In [ ]:
# Install dbt-databricks
%pip install dbt-databricks --quiet

In [ ]:
# Create dbt project
import os
import subprocess

DBT_PROJECT_PATH = "/tmp/dbt_project"
os.makedirs(DBT_PROJECT_PATH, exist_ok=True)
os.chdir(DBT_PROJECT_PATH)

# Create dbt_project.yml
dbt_project_yml = """
name: 'food_delivery_analytics'
version: '1.0.0'
config-version: 2

profile: 'food_delivery'

model-paths: ["models"]
test-paths: ["tests"]
macro-paths: ["macros"]

target-path: "target"
clean-targets: ["target", "dbt_packages"]

models:
  food_delivery_analytics:
    staging:
      +materialized: view
    intermediate:
      +materialized: ephemeral
    marts:
      +materialized: table
"""

with open(f"{DBT_PROJECT_PATH}/dbt_project.yml", "w") as f:
    f.write(dbt_project_yml)

# Create profiles.yml
profiles_yml = """
food_delivery:
  target: dev
  outputs:
    dev:
      type: databricks
      host: ${env:DATABRICKS_HOST}
      http_path: ${env:DATABRICKS_HTTP_PATH}
      token: ${env:DATABRICKS_TOKEN}
      schema: food_delivery_analytics
      threads: 4
"""

os.makedirs(f"{DBT_PROJECT_PATH}/.dbt", exist_ok=True)
with open(f"{DBT_PROJECT_PATH}/profiles.yml", "w") as f:
    f.write(profiles_yml)

print("Created dbt project files")

In [ ]:
# Create directory structure
os.makedirs(f"{DBT_PROJECT_PATH}/models/staging", exist_ok=True)
os.makedirs(f"{DBT_PROJECT_PATH}/models/intermediate", exist_ok=True)
os.makedirs(f"{DBT_PROJECT_PATH}/models/marts", exist_ok=True)
os.makedirs(f"{DBT_PROJECT_PATH}/macros", exist_ok=True)
os.makedirs(f"{DBT_PROJECT_PATH}/tests", exist_ok=True)

# Create macros
with open(f"{DBT_PROJECT_PATH}/macros/datediff_minutes.sql", "w") as f:
    f.write("""
{% macro datediff_minutes(start_col, end_col) %}
    (epoch({{ end_col }}) - epoch({{ start_col }})) / 60
{% endmacro %}
""")

# Create sources.yml
with open(f"{DBT_PROJECT_PATH}/models/sources.yml", "w") as f:
    f.write("""
version: 2

sources:
  - name: silver
    description: "Silver layer tables"
    tables:
      - name: silver_order_facts
      - name: silver_delivery_ops
      - name: silver_restaurant_support
""")

print("Created directory structure")

In [ ]:
# Create staging models

# stg_orders.sql
with open(f"{DBT_PROJECT_PATH}/models/staging/stg_orders.sql", "w") as f:
    f.write("""
{{ config(materialized='view') }}

select
    order_id,
    customer_id,
    restaurant_id,
    city,
    status,
    payment_mode,
    time_slot,
    order_ts,
    promised_delivery_ts,
    order_date,
    order_hour,
    is_weekend,
    order_value,
    item_count,
    items_subtotal,
    has_refund,
    refund_amount
from {{ source('silver', 'silver_order_facts') }}
where _dq_invalid_order_value = false
""")

# stg_delivery_events.sql
with open(f"{DBT_PROJECT_PATH}/models/staging/stg_delivery_events.sql", "w") as f:
    f.write("""
{{ config(materialized='view') }}

select
    order_id,
    rider_id,
    rider_city,
    event_type,
    event_ts,
    latitude,
    longitude
from {{ source('silver', 'silver_delivery_ops') }}
where _dq_orphan_order = false
""")

# stg_restaurants.sql
with open(f"{DBT_PROJECT_PATH}/models/staging/stg_restaurants.sql", "w") as f:
    f.write("""
{{ config(materialized='view') }}

select distinct
    restaurant_id,
    city as restaurant_city,
    cuisine_type,
    rating_band,
    onboarding_date
from {{ source('silver', 'silver_restaurant_support') }}
where restaurant_id is not null
""")

print("Created staging models")

In [ ]:
# Create intermediate model

with open(f"{DBT_PROJECT_PATH}/models/intermediate/int_order_delivery_timeline.sql", "w") as f:
    f.write("""
{{ config(materialized='ephemeral') }}

with orders as (
    select * from {{ ref('stg_orders') }}
),

events as (
    select * from {{ ref('stg_delivery_events') }}
),

event_pivot as (
    select
        order_id,
        max(case when event_type = 'order_confirmed' then event_ts end) as confirmed_ts,
        max(case when event_type = 'restaurant_accepted' then event_ts end) as accepted_ts,
        max(case when event_type = 'food_ready' then event_ts end) as food_ready_ts,
        max(case when event_type = 'rider_picked_up' then event_ts end) as picked_up_ts,
        max(case when event_type = 'delivered' then event_ts end) as delivered_ts,
        max(rider_id) as rider_id
    from events
    group by order_id
),

final as (
    select
        o.order_id,
        o.restaurant_id,
        o.city,
        o.order_ts,
        o.promised_delivery_ts,
        o.status,
        o.order_value,
        o.time_slot,
        o.order_date,
        o.is_weekend,
        ep.confirmed_ts,
        ep.accepted_ts,
        ep.food_ready_ts,
        ep.picked_up_ts,
        ep.delivered_ts,
        ep.rider_id,
        {{ datediff_minutes('ep.accepted_ts', 'ep.food_ready_ts') }} as prep_time_minutes,
        {{ datediff_minutes('ep.picked_up_ts', 'ep.delivered_ts') }} as delivery_time_minutes,
        {{ datediff_minutes('o.order_ts', 'ep.delivered_ts') }} as total_time_minutes,
        case when ep.delivered_ts is null then null
             when ep.delivered_ts > o.promised_delivery_ts then true
             else false end as is_sla_breached
    from orders o
    left join event_pivot ep on o.order_id = ep.order_id
)

select * from final
""")

print("Created intermediate model")

In [ ]:
# Create marts

# fct_orders.sql
with open(f"{DBT_PROJECT_PATH}/models/marts/fct_orders.sql", "w") as f:
    f.write("""
{{ config(materialized='table') }}

select * from {{ ref('int_order_delivery_timeline') }}
""")

# mart_sla_breach_analysis.sql
with open(f"{DBT_PROJECT_PATH}/models/marts/mart_sla_breach_analysis.sql", "w") as f:
    f.write("""
{{ config(materialized='table', description='SLA breach analysis by city and time slot') }}

with order_timeline as (
    select * from {{ ref('int_order_delivery_timeline') }}
    where status = 'delivered'
),

metrics as (
    select
        city,
        time_slot,
        count(*) as total_orders,
        sum(case when is_sla_breached then 1 else 0 end) as breached_orders,
        round(avg(total_time_minutes), 1) as avg_delivery_time_minutes
    from order_timeline
    group by city, time_slot
)

select
    city,
    time_slot,
    total_orders,
    breached_orders,
    round(breached_orders * 100.0 / total_orders, 2) as breach_rate_pct,
    avg_delivery_time_minutes
from metrics
order by breach_rate_pct desc
""")

# mart_weekly_trends.sql
with open(f"{DBT_PROJECT_PATH}/models/marts/mart_weekly_trends.sql", "w") as f:
    f.write("""
{{ config(materialized='table', description='Weekly business trends') }}

with weekly as (
    select
        date_trunc('week', order_date) as week_start,
        count(*) as total_orders,
        sum(case when status = 'delivered' then 1 else 0 end) as completed_orders,
        sum(case when status = 'cancelled' then 1 else 0 end) as cancelled_orders,
        round(sum(order_value), 2) as gmv
    from {{ ref('fct_orders') }}
    group by date_trunc('week', order_date)
)

select * from weekly order by week_start desc
""")

print("Created marts")

In [ ]:
# Run dbt
import os
os.environ["DBT_PROFILES_DIR"] = DBT_PROJECT_PATH

print("\n" + "="*60)
print("Running dbt models")
print("="*60)

# Check environment
host = os.environ.get("DATABRICKS_HOST", "")
http_path = os.environ.get("DATABRICKS_HTTP_PATH", "")
token = os.environ.get("DATABRICKS_TOKEN", "")

if not host or not http_path or not token:
    print("\n⚠️  WARNING: Databricks credentials not set!")
    print("Please set these environment variables:")
    print("  os.environ['DATABRICKS_HOST'] = 'https://your-workspace.cloud.databricks.com'")
    print("  os.environ['DATABRICKS_HTTP_PATH'] = '/sql/1.0/warehouses/your-warehouse-id'")
    print("  os.environ['DATABRICKS_TOKEN'] = 'your-token'")
    print("\nFor now, creating views directly in Spark...")
else:
    print(f"Connected to: {host}")
    print(f"Warehouse: {http_path}")

In [ ]:
# Create views directly in Spark (if dbt not configured)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, max as spark_max, sum as spark_sum, count, round as spark_round, avg

spark = SparkSession.builder.getOrCreate()

print("\n--- Creating Analytics Views ---")

# Create stg_orders view
spark.sql("""
CREATE OR REPLACE VIEW stg_orders AS
SELECT
    order_id, customer_id, restaurant_id, city, status, payment_mode,
    time_slot, order_ts, promised_delivery_ts, order_date, order_hour,
    is_weekend, order_value, item_count, items_subtotal, has_refund, refund_amount
FROM silver_order_facts
WHERE _dq_invalid_order_value = false
""")
print("✓ stg_orders view created")

# Create stg_delivery_events view
spark.sql("""
CREATE OR REPLACE VIEW stg_delivery_events AS
SELECT order_id, rider_id, rider_city, event_type, event_ts, latitude, longitude
FROM silver_delivery_ops
WHERE _dq_orphan_order = false
""")
print("✓ stg_delivery_events view created")

# Create stg_restaurants view
spark.sql("""
CREATE OR REPLACE VIEW stg_restaurants AS
SELECT DISTINCT restaurant_id, city as restaurant_city, cuisine_type, rating_band, onboarding_date
FROM silver_restaurant_support
WHERE restaurant_id IS NOT NULL
""")
print("✓ stg_restaurants view created")

In [ ]:
# Create fct_orders table
print("\n--- Creating Fact Tables ---")

spark.sql("""
CREATE OR REPLACE TABLE fct_orders AS
SELECT
    o.order_id, o.restaurant_id, o.city, o.order_ts, o.promised_delivery_ts,
    o.status, o.order_value, o.time_slot, o.order_date, o.is_weekend,
    e.confirmed_ts, e.accepted_ts, e.food_ready_ts, e.picked_up_ts, e.delivered_ts,
    e.rider_id, e.prep_time_minutes, e.delivery_time_minutes, e.total_time_minutes,
    e.is_sla_breached
FROM stg_orders o
LEFT JOIN (
    SELECT
        order_id,
        MAX(CASE WHEN event_type = 'order_confirmed' THEN event_ts END) AS confirmed_ts,
        MAX(CASE WHEN event_type = 'restaurant_accepted' THEN event_ts END) AS accepted_ts,
        MAX(CASE WHEN event_type = 'food_ready' THEN event_ts END) AS food_ready_ts,
        MAX(CASE WHEN event_type = 'rider_picked_up' THEN event_ts END) AS picked_up_ts,
        MAX(CASE WHEN event_type = 'delivered' THEN event_ts END) AS delivered_ts,
        MAX(rider_id) AS rider_id,
        (UNIX_TIMESTAMP(MAX(CASE WHEN event_type = 'food_ready' THEN event_ts END)) - 
         UNIX_TIMESTAMP(MAX(CASE WHEN event_type = 'restaurant_accepted' THEN event_ts END))) / 60 AS prep_time_minutes,
        (UNIX_TIMESTAMP(MAX(CASE WHEN event_type = 'delivered' THEN event_ts END)) - 
         UNIX_TIMESTAMP(MAX(CASE WHEN event_type = 'rider_picked_up' THEN event_ts END))) / 60 AS delivery_time_minutes,
        (UNIX_TIMESTAMP(MAX(CASE WHEN event_type = 'delivered' THEN event_ts END)) - 
         UNIX_TIMESTAMP(order_ts)) / 60 AS total_time_minutes,
        CASE WHEN MAX(CASE WHEN event_type = 'delivered' THEN event_ts END) > promised_delivery_ts THEN true ELSE false END AS is_sla_breached
    FROM stg_delivery_events
    GROUP BY order_id, order_ts, promised_delivery_ts
) e ON o.order_id = e.order_id
""")
print("✓ fct_orders table created")

In [ ]:
# Create mart_sla_breach_analysis
spark.sql("""
CREATE OR REPLACE TABLE mart_sla_breach_analysis AS
SELECT
    city, time_slot,
    COUNT(*) as total_orders,
    SUM(CASE WHEN is_sla_breached THEN 1 ELSE 0 END) as breached_orders,
    ROUND(SUM(CASE WHEN is_sla_breached THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as breach_rate_pct,
    ROUND(AVG(total_time_minutes), 1) as avg_delivery_time_minutes
FROM fct_orders
WHERE status = 'delivered'
GROUP BY city, time_slot
ORDER BY breach_rate_pct DESC
""")
print("✓ mart_sla_breach_analysis table created")

# Create mart_weekly_trends
spark.sql("""
CREATE OR REPLACE TABLE mart_weekly_trends AS
SELECT
    DATE_TRUNC('week', order_date) as week_start,
    COUNT(*) as total_orders,
    SUM(CASE WHEN status = 'delivered' THEN 1 ELSE 0 END) as completed_orders,
    SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) as cancelled_orders,
    ROUND(SUM(order_value), 2) as gmv
FROM fct_orders
GROUP BY DATE_TRUNC('week', order_date)
ORDER BY week_start DESC
""")
print("✓ mart_weekly_trends table created")

In [ ]:
# Summary
print("\n" + "="*60)
print("DBT MODELS COMPLETE")
print("="*60)

tables = ["fct_orders", "mart_sla_breach_analysis", "mart_weekly_trends"]
views = ["stg_orders", "stg_delivery_events", "stg_restaurants"]

print("\nTables created:")
for t in tables:
    count = spark.table(t).count()
    print(f"  {t}: {count:,} rows")

print("\nViews created:")
for v in views:
    print(f"  {v}")

spark.stop()